In [ ]:
from datasets import load_dataset, DatasetDict
hf_nature_ds = load_dataset("mertcobanov/nature-dataset")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
split = hf_nature_ds["train"].train_test_split(test_size=0.3, seed=42)

trainSet = split["train"]
testSet = split["test"]

from datasets import load_dataset, DatasetDict
import re

NON_TRAVERSIBLE_SUBJECTS = {
    # Water bodies
    "ocean", "lake", "river", "flood", "lava", "quicksand",
    "glacier", "chasm", "sea", "waterfall", "pond", "stream",
    "brook", "creek", "reservoir", "lagoon", "bay", "gulf",
    "inlet", "estuary", "delta", "fjord", "strait", "channel",
    "rapids", "cascade", "torrent", "whirlpool", "tide", "surf",
    "wave", "tsunami", "puddle", "marsh", "bog", "fen", "mire",
    "quagmire", "swamp", "wetland", "floodplain", "moat",
    # Drops & hazardous elevation
    "cliff", "ravine", "gorge", "canyon", "precipice", "abyss",
    "crevasse", "crevice", "fissure", "crater", "pit", "sinkhole",
    "escarpment", "bluff", "ledge", "overhang", "scarp", "dropoff",
    "chute", "gulch", "gully", "trench", "void", "shaft",
    # Dense vegetation
    "thicket", "bramble", "undergrowth", "briars", "thorns",
    "scrub", "brush", "tangle", "briar", "nettle", "shrub",
    "hedge", "canopy", "jungle", "rainforest", "mangrove",
    "bracken", "fern", "sedge", "reed", "bulrush",
    # Unstable ground
    "mud", "lava", "magma", "ash", "scree", "talus", "gravel",
    "rubble", "debris", "landslide", "avalanche", "rockfall",
    "mudslide", "erosion", "subsidence",
    # Structures & barriers
    "wall", "fence", "barrier", "barricade", "dam", "weir",
    "embankment", "levee", "dyke", "rampart", "fortification",
}

TRAVERSIBLE_INDICATORS = {
    # Paths & trails
    "path", "trail", "track", "route", "walkway", "footpath",
    "towpath", "bridleway", "byway", "lane", "alley", "passage",
    "corridor", "shortcut", "detour", "circuit", "loop",
    # Roads & paved surfaces
    "road", "street", "avenue", "boulevard", "highway", "motorway",
    "freeway", "expressway", "carriageway", "driveway", "pavement",
    "sidewalk", "tarmac", "asphalt", "cobblestone", "flagstone",
    "paving", "causeway", "thoroughfare", "beltway", "turnpike",
    # Crossings & connections
    "bridge", "overpass", "underpass", "viaduct", "flyover",
    "crossing", "ford", "stepping", "jetty", "pier",
    "pontoon", "gangway", "ramp", "junction", "interchange",
    # Vertical traversal
    "steps", "stairs", "stairway", "staircase", "ladder", "ramp",
    "slope", "gradient", "ascent", "descent", "incline", "decline",
    # Open traversible terrain
    "meadow", "field", "plain", "plateau", "clearing", "glade",
    "lawn", "grassland", "pasture", "paddock", "common", "heath",
    "moor", "tundra", "savanna", "prairie", "steppe", "flat",
    # Marked routes
    "waypoint", "marker", "signpost", "waymarked", "blazed",
    "mapped", "designated", "maintained", "groomed", "paved",
}

CONDITIONAL_BLOCKERS = {
    # Weather & surface conditions
    "water", "steep", "snow", "ice", "frost", "sleet", "hail",
    "frozen", "slippery", "icy", "wet", "waterlogged", "sodden",
    # Terrain descriptors
    "rocky", "dense", "overgrown", "rugged", "rough", "uneven",
    "jagged", "boulder", "boulders", "stony", "loose", "unstable",
    "crumbling", "eroded", "washed",
    # Vegetation density
    "forested", "wooded", "bushy", "tangled", "matted", "thick",
    # Elevation & gradient
    "mountain", "mountainous", "highland", "upland", "alpine",
    "vertical", "sheer", "exposed",
}

def labelImage(caption):
  caption = caption.lower()
  words = set(re.findall(r'\b\w+\b', caption))
  has_traversible = bool(words & TRAVERSIBLE_INDICATORS)
  has_hard_blocker = bool(words & NON_TRAVERSIBLE_SUBJECTS)
  has_conditional = bool(words & CONDITIONAL_BLOCKERS)

  is_not_traversible = has_hard_blocker or (has_conditional and not has_traversible)
  if is_not_traversible:
    return 0
  else:
    return 1





In [ ]:
print(trainSet[0]["image"])

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x785678FC1370>


In [ ]:
import numpy as np
from google.colab import files
from PIL import Image

TARGET_SIZE = (96, 96)

train_ds_x = np.zeros((len(trainSet), TARGET_SIZE[0], TARGET_SIZE[1]), dtype = np.float32)
test_ds_x = np.zeros((len(testSet), TARGET_SIZE[0], TARGET_SIZE[1]), dtype = np.float32)

train_ds_y = np.zeros((len(trainSet)), dtype = np.int8)
test_ds_y = np.zeros((len(testSet)), dtype = np.int8)

def processDataset(xSet, ySet, df):
  for i in range(len(df)):
    image = df[i]["image"]
    label = labelImage(df[i]["caption"])

    preprocessedImage = image.convert("L").resize(TARGET_SIZE)
    xSet[i] = np.array(preprocessedImage)
    ySet[i] = label

  return xSet, ySet

finalTrainDsX, finalTrainDsY = processDataset(train_ds_x, train_ds_y, trainSet)
finalTestDsX, finalTestDsY = processDataset(test_ds_x, test_ds_y, testSet)

np.savez("trainDatasetNature.npz", x=finalTrainDsX, y=finalTrainDsY)
np.savez("testDatasetNature.npz", x=finalTestDsX, y=finalTestDsY)

files.download("trainDatasetNature.npz")
files.download("testDatasetNature.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>